# MNIST dense network — Solution to the exercises

*Exposome Analytics Summer School, London 2026 — Deep Learning day.*

Answers to the exercises of **A Dense Neural Network on MNIST — a guided tour**.
Width, depth, dropout, learning rate and activation are compared under the
**same budget**, so the numbers can be put side by side.

> **Switch to a GPU first:** Runtime → Change runtime type → T4 GPU.


## Setup, data, and the two helpers


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])

full_train = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set   = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

g = torch.Generator().manual_seed(0)
train_set, val_set = random_split(full_train, [48000, 12000], generator=g)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256)
test_loader  = DataLoader(test_set,  batch_size=256)


In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=(256, 128), dropout=0.2, act=F.relu):
        super().__init__()
        self.act = act
        dims = [28 * 28] + list(hidden)
        self.layers = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.out = nn.Linear(dims[-1], 10)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        for layer in self.layers:
            x = self.drop(self.act(layer(x)))
        return self.out(x)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss_sum += F.cross_entropy(logits, labels, reduction='sum').item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum / total, correct / total


## One run, with early stopping

Up to 20 epochs, and we keep the weights of the epoch with the best validation
loss. Patience of 3.


In [ ]:
import copy

def run(name, hidden=(256, 128), dropout=0.2, act=F.relu, lr=1e-3, epochs=20, patience=3):
    torch.manual_seed(0)
    model = MLP(hidden, dropout, act).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss, best_state, bad, used = float('inf'), None, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            F.cross_entropy(model(images), labels).backward()
            opt.step()
        val_loss, _ = evaluate(model, val_loader)
        used = epoch
        if val_loss < best_loss - 1e-4:
            best_loss, bad = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    _, test_acc = evaluate(model, test_loader)
    n_par = sum(p.numel() for p in model.parameters())
    print(f'{name:32s} test acc {test_acc:.4f}   ({used:2d} epochs, {n_par:,} params)')
    return {'configuration': name, 'test accuracy': round(test_acc, 4),
            'epochs used': used, 'parameters': n_par}


## The experiments

Exercises 1 to 5 of the guided tour, one line each.


In [ ]:
results = []
results.append(run('baseline 256-128, drop 0.2'))
results.append(run('wider    512-256',        hidden=(512, 256)))
results.append(run('deeper   256-128-64',     hidden=(256, 128, 64)))
results.append(run('no dropout',              dropout=0.0))
results.append(run('dropout 0.5',             dropout=0.5))
results.append(run('lr 1e-2',                 lr=1e-2))
results.append(run('lr 1e-4',                 lr=1e-4))
results.append(run('tanh instead of relu',    act=torch.tanh))


## The table


In [ ]:
table = (pd.DataFrame(results)
           .sort_values('test accuracy', ascending=False)
           .reset_index(drop=True))
table


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = table.sort_values('test accuracy')
ax.barh(order['configuration'], order['test accuracy'], color='steelblue')
ax.set_xlim(0.93, 1.0)
ax.set_xlabel('test accuracy')
ax.set_title('Same budget for every configuration')
for i, v in enumerate(order['test accuracy']):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()


---

## Answers, exercise by exercise

**1. Width.** Going from `256-128` to `512-256` buys very little, for roughly
four times the parameters. The baseline was already wide enough for MNIST.

**2. Depth.** A third layer changes almost nothing here. Depth pays when there
is a hierarchy to build; on 784 flattened pixels there is not much of one.

**3. Dropout.** With `0.0` the training loss falls faster and the gap with the
validation loss opens sooner: that is overfitting, visible in the number of
epochs used before early stopping fires. With `0.5` the network is slowed down
and, at this size, regularised more than it needs.

**4. Learning rate.** `1e-2` is unstable, the validation loss jumps about and
early stopping may cut in on noise. `1e-4` is stable but slow: it would need far
more than 20 epochs. This is the single most sensitive knob of the whole
notebook.

**5. Activation.** `tanh` still works, a little below `relu`. It saturates at
both ends, so the gradient vanishes for large activations — the reason `relu`
took over.

**7. The challenge.** A convolutional network passes 99% with fewer parameters
than the baseline here, because it uses the fact that the input is an image.
Flattening the 28×28 grid into 784 numbers throws that away on the first line.
